In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

load_dotenv()
data_pth = Path(os.getenv("DATA_PATH", "data"))
train_csv_pth = f"{data_pth}/train.csv"
train_img_pth = Path(f"{data_pth}/train_images")

train_csv = pd.read_csv(train_csv_pth)
train_csv = train_csv.copy()

In [ ]:
imageide = train_csv["ImageId"].value_counts()
duplicates_combined = imageide.index.tolist()
train_file_name = [
    file_path.name for file_path in train_img_pth.iterdir() if file_path.is_file()
]

# Grouped defect profiles
class_ = []
for item in duplicates_combined:
    duplicate_statuses = train_csv[["ImageId", "ClassId"]].loc[
        train_csv["ImageId"] == item
    ]
    class_.append((item, "&".join(map(str, duplicate_statuses.ClassId.tolist()))))

# Matched train.csv with train_images and assigned "Clean" profile to train_images without defects
for obj in train_file_name:
    if obj not in duplicates_combined:
        class_.append((obj, "Clean"))

In [ ]:
len(train_file_name)

In [ ]:
import csv

headers = ["ImageId", "Class"]
file_pth = Path(os.getenv("CURRENT_FILE_PATH", ""))
with open("train_folds.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(headers)
    writer.writerows(class_)

In [ ]:
train_fold = pd.read_csv("train_folds.csv")
train_fold["Fold"] = -1

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(train_fold, train_fold["Class"])):
    train_fold.loc[val_idx, "Fold"] = fold

train_fold.to_csv("train_folds.csv", index=False)

In [ ]:
def split_df_on_fold(
    val_fold_no: int,
    train_fold_df: pd.DataFrame,
    train_csv_df: pd.DataFrame,
):
    train_fold = train_fold_df.copy()
    train_csv = train_csv_df.copy()

    clean = train_fold[["ImageId", "ClassId"]].loc[train_fold["ClassId"] == "Clean"]
    concat_train_csv = pd.concat([train_csv, clean], ignore_index=True)
    concat_train_csv = concat_train_csv.merge(train_fold, how="inner", on="ImageId")
    train_df = concat_train_csv.loc[concat_train_csv["Fold"] == val_fold_no]
    val_df = concat_train_csv.loc[concat_train_csv["Fold"] != val_fold_no]
    return train_df, val_df

In [ ]:
train_fold.rename(columns={"Class": "ClassId"}, inplace=True)

In [ ]:
clean = train_fold[["ImageId", "ClassId"]].loc[train_fold["ClassId"] == "Clean"]
concat_train_csv = pd.concat([train_csv, clean], ignore_index=True)
concat_train_csv.tail()

In [ ]:
concat_train_csv = concat_train_csv.merge(train_fold, how="inner", on="ImageId")
concat_train_csv.head()

In [ ]:
train_df = concat_train_csv.loc[concat_train_csv["Fold"] == 1]
train_df

In [ ]:
val_df = concat_train_csv.loc[concat_train_csv["Fold"] != 1]
val_df